In [30]:
import pandas as pd
import numpy as np

In [31]:
df = pd.read_csv("../data/bronze/Entities.csv", index_col=0)

Limpieza general

In [32]:
df = df.drop(columns=["property_id","agency","agent","location_id","page_url"])

In [33]:
print(len(df))
df = df.drop_duplicates()
print(len(df))

168446
147957


Limpieza Variables:

Primera vuelta limpieza: 

Variable baths

In [34]:
df["baths"].value_counts()

baths
0      35572
3      29060
2      22449
4      17985
6      15362
5      14067
7       5151
1       4792
8       1970
10       839
9        688
12        10
11         5
13         3
14         3
403        1
Name: count, dtype: int64

In [35]:
df = df[df["baths"] < 7]

In [36]:
len(df)

139287

Variable bedrooms: 

In [37]:
df["bedrooms"].value_counts()

bedrooms
3     38996
2     26764
4     21181
5     19768
0     18135
6      8218
1      3925
7       904
8       631
9       326
10      308
11       80
12       21
14        7
15        7
16        4
28        3
25        3
13        2
18        1
21        1
68        1
27        1
Name: count, dtype: int64

In [38]:
df = df[df["bedrooms"]<7]
len(df)

136987

Variables bedroom - bathroom conjuntas: 

In [39]:
df.groupby("bedrooms")["baths"].value_counts().unstack(fill_value=0)

baths,0,1,2,3,4,5,6
bedrooms,,,,,,,
0,17817,74,89,61,31,27,36
1,530,2890,485,14,3,2,1
2,3679,1542,18415,2948,141,21,18
3,5519,226,2406,23719,6428,654,44
4,2874,40,795,1325,9966,5228,953
5,2532,8,153,572,811,7388,8304
6,1777,7,80,304,362,468,5220


In [40]:
df = df[~((df["bedrooms"] == 0) & (df["baths"] > 0))].reset_index(drop=True)

In [41]:
df = df[~((df["baths"] == 0) & (df["bedrooms"] > 1))].reset_index(drop=True)

In [42]:
n_antes = len(df)
df = df.query("not (bedrooms == 0 and baths == 0)").reset_index(drop=True)
print(f"Filas eliminadas: {n_antes - len(df)}")

Filas eliminadas: 17817


Variable city:

In [43]:
df["city"].value_counts()

city
Karachi       38185
Lahore        27453
Islamabad     22790
Rawalpindi    11754
Faisalabad     2289
Name: count, dtype: int64

Variable date_added: 

In [44]:
df["date_added"] = pd.to_datetime(df["date_added"])
df["date_added"].dtype

dtype('<M8[ns]')

Variable location: 

In [45]:
df["location"].nunique()

1406

In [46]:
CENTROS = {
    "Lahore":     (31.582045, 74.329376),
    "Karachi":    (24.860966, 66.990501),
    "Islamabad":  (33.738045, 73.084488),
    "Rawalpindi": (33.626057, 73.071442),
    "Faisalabad": (31.418715, 73.079109),
}
R_TIERRA_KM = 6371.0088

lat_c = df["city"].map(lambda c: CENTROS[c][0])
lon_c = df["city"].map(lambda c: CENTROS[c][1])

lat1, lon1 = np.radians(df["latitude"]), np.radians(df["longitude"])
lat2, lon2 = np.radians(lat_c), np.radians(lon_c)

# Fórmula de Haversine (distancia en línea recta sobre la esfera, en km)
a = np.sin((lat1 - lat2) / 2) ** 2 + np.cos(lat2) * np.cos(lat1) * np.sin((lon1 - lon2) / 2) ** 2
df["dist_centro_km"] = (2 * R_TIERRA_KM * np.arcsin(np.sqrt(a))).round(2)

df.groupby("city")["dist_centro_km"].describe()

,count,mean,std,min,25%,50%,75%,max
city,,,,,,,,
Faisalabad,2289.0,7.954688,82.148318,0.00,4.04,5.61,7.36,3932.06
Islamabad,22790.0,14.422144,54.921898,0.72,9.38,11.53,18.90,5404.75
Karachi,38185.0,15.453015,13.660381,0.30,8.86,12.23,18.10,942.89
Lahore,27453.0,16.098127,7.386394,0.32,11.92,15.30,19.05,268.65
Rawalpindi,11754.0,9.842999,37.558160,0.27,6.92,8.91,12.55,4046.89


In [47]:
n_antes = len(df)
df = df[df["dist_centro_km"] <= 100].reset_index(drop=True)
print(f"Filas eliminadas por coordenadas erróneas: {n_antes - len(df)}")

Filas eliminadas por coordenadas erróneas: 23


Variable price

In [48]:
PKR_TO_EUR_2019 = 1 / 179.61   # ≈ 0.005568, valutafx, 18/07/2019

df["price"] = (df["price"] * PKR_TO_EUR_2019).round(2)

Variable property type

In [49]:
df["property_type"].value_counts()

property_type
House            60483
Flat             26951
Upper Portion     8528
Lower Portion     5514
Room               494
Penthouse          270
Farm House         208
Name: count, dtype: int64

In [50]:
df[df["property_type"] == "Room"]

,property_type,price,location,city,province_name,latitude,longitude,baths,purpose,bedrooms,date_added,Total_Area,dist_centro_km
156,Room,44.54,Gulshan-e-Iqbal Town,Karachi,Sindh,24.898465,67.062500,1,For Rent,1,2019-07-18,26136.096,8.37
419,Room,278.38,DHA Defence,Karachi,Sindh,24.830406,67.053764,3,For Rent,3,2019-07-18,5445.020,7.23
675,Room,83.51,G-11,Islamabad,Islamabad Capital,33.663818,72.995260,1,For Rent,1,2019-06-27,4356.016,11.67
1278,Room,128.06,DHA Defence,Karachi,Sindh,24.834337,67.055397,1,For Rent,1,2019-07-18,19329.821,7.19
1557,Room,111.35,G-11,Islamabad,Islamabad Capital,33.669533,73.000073,0,For Rent,1,2019-07-17,4900.518,10.91
...,...,...,...,...,...,...,...,...,...,...,...,...,...
101232,Room,306.22,DHA Defence,Karachi,Sindh,24.782215,67.074537,1,For Rent,1,2019-07-17,5445.000,12.19
101569,Room,33.41,Dhoke Banaras Road,Rawalpindi,Punjab,33.592470,73.014950,0,For Rent,1,2019-07-17,272.251,6.43
101695,Room,178.16,DHA Defence,Karachi,Sindh,24.805045,67.064323,1,For Rent,1,2019-07-17,11979.044,9.70
101834,Room,445.41,DHA Defence,Karachi,Sindh,24.805045,67.064323,1,For Rent,1,2019-07-17,5445.000,9.70


In [51]:
mask = (df["bedrooms"] > 1) & (df["property_type"] == "Room")
print(f"Filas que cambian de Room a Flat: {mask.sum()}")

df.loc[mask, "property_type"] = "Flat"

Filas que cambian de Room a Flat: 64


Variable purpose:

In [52]:
df["purpose"].value_counts()

purpose
For Sale    74032
For Rent    28416
Name: count, dtype: int64

In [53]:
df = df[df["purpose"] == "For Sale"]
df = df.drop(columns=["purpose"]).reset_index(drop=True)

In [54]:
df

,property_type,price,location,city,province_name,latitude,longitude,baths,bedrooms,date_added,Total_Area,dist_centro_km
0,Flat,55676.19,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,2,2019-02-04,1089.004,9.27
1,Flat,38416.57,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,3,2019-05-04,15246.056,11.23
2,House,91865.71,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,5,2019-07-17,2178.008,18.81
3,House,242191.41,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,4,2019-04-05,10890.000,7.04
4,House,38973.33,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,3,2019-07-10,2178.008,33.88
...,...,...,...,...,...,...,...,...,...,...,...,...
74027,Flat,28951.62,Bahria Town Karachi,Karachi,Sindh,25.013015,67.314401,2,2,2019-07-18,11434.542,36.78
74028,Flat,41757.14,Bahria Town Karachi,Karachi,Sindh,25.013015,67.314401,3,3,2019-07-18,2178.008,36.78
74029,House,48995.04,Bahria Town Karachi,Karachi,Sindh,25.071948,67.338656,4,3,2019-07-18,2178.008,42.21
74030,House,77946.66,Bahria Town Karachi,Karachi,Sindh,25.058062,67.306626,3,3,2019-07-18,2178.008,38.68


Variable Total_Area: 

In [55]:
df["Total_Area"] = (df["Total_Area"] * 0.09290304).round(2)

Segunda vuelta limpieza: 

Variable property_type

In [56]:
df = df[~((df["property_type"] == "Flat") & (df["Total_Area"] > 250))].reset_index(drop=True)

In [57]:
n_antes = len(df)
df = df[~((df["property_type"] == "House") & (df["Total_Area"] > 2500))].reset_index(drop=True)
print(f"Casas eliminadas: {n_antes - len(df)}")

Casas eliminadas: 2078


In [58]:
df

,property_type,price,location,city,province_name,latitude,longitude,baths,bedrooms,date_added,Total_Area,dist_centro_km
0,Flat,55676.19,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,2,2019-02-04,101.17,9.27
1,House,91865.71,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,5,2019-07-17,202.34,18.81
2,House,242191.41,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,4,2019-04-05,1011.71,7.04
3,House,38973.33,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,3,2019-07-10,202.34,33.88
4,Penthouse,222704.75,F-11,Islamabad,Islamabad Capital,33.679211,72.988787,5,5,2019-06-27,505.86,11.01
...,...,...,...,...,...,...,...,...,...,...,...,...
55701,House,77946.66,Bahria Town Karachi,Karachi,Sindh,25.005740,67.316385,5,4,2019-07-18,354.10,36.59
55702,Flat,41757.14,Bahria Town Karachi,Karachi,Sindh,25.013015,67.314401,3,3,2019-07-18,202.34,36.78
55703,House,48995.04,Bahria Town Karachi,Karachi,Sindh,25.071948,67.338656,4,3,2019-07-18,202.34,42.21
55704,House,77946.66,Bahria Town Karachi,Karachi,Sindh,25.058062,67.306626,3,3,2019-07-18,202.34,38.68


In [59]:
df["city"].value_counts()

city
Lahore        26067
Karachi       15647
Islamabad      6423
Rawalpindi     6028
Faisalabad     1541
Name: count, dtype: int64